# E-Commerce En Accion: EDA Guiado Curie

![E-commerce](https://images.unsplash.com/photo-1556740714-a8395b3bf30f?auto=format&fit=crop&w=1200&q=60)

Aqui el dataset protagonista es comercio electronico (modo Kaggle Olist o simulado).
Este cuaderno te guia paso a paso hasta insights accionables.

# Curie (Guiado): EDA Completo Paso a Paso

Objetivo: ejecutar un EDA completo de forma consistente y rapida.

Modo de uso:
- `simulado`: ideal para clase rapida
- `kaggle`: ideal para caso real (Olist)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', None)

## Paso 1: Elegir fuente de datos

In [ ]:
MODO = 'simulado'  # cambia a 'kaggle' si tienes los CSV de Olist

if MODO == 'kaggle':
    archivos = [
        'olist_customers_dataset.csv',
        'olist_orders_dataset.csv',
        'olist_order_items_dataset.csv',
        'olist_order_payments_dataset.csv',
        'olist_products_dataset.csv'
    ]

    faltantes = [f for f in archivos if not os.path.exists(f)]
    if faltantes:
        print('Faltan archivos Kaggle, se usa modo simulado.')
        MODO = 'simulado'

if MODO == 'kaggle':
    clientes = pd.read_csv('olist_customers_dataset.csv')
    ordenes = pd.read_csv('olist_orders_dataset.csv')
    items = pd.read_csv('olist_order_items_dataset.csv')
    pagos = pd.read_csv('olist_order_payments_dataset.csv')
    productos = pd.read_csv('olist_products_dataset.csv')

    df = items.merge(ordenes, on='order_id', how='left') \
              .merge(pagos.groupby('order_id', as_index=False)['payment_value'].sum(), on='order_id', how='left') \
              .merge(productos[['product_id', 'product_category_name']], on='product_id', how='left')
else:
    np.random.seed(42)
    n = 500
    df = pd.DataFrame({
        'cliente_id': range(1, n + 1),
        'edad': np.random.randint(18, 70, n),
        'genero': np.random.choice(['M', 'F'], n),
        'region': np.random.choice(['Norte', 'Sur', 'Este', 'Oeste'], n),
        'categoria': np.random.choice(['Electronica', 'Ropa', 'Hogar'], n),
        'precio': np.random.normal(100, 50, n),
        'cantidad': np.random.randint(1, 10, n),
        'descuento': np.random.choice([0, 5, 10, 15, 20], n),
    })
    df['precio'] = np.maximum(df['precio'], 10)
    df['total'] = df['precio'] * df['cantidad'] * (1 - df['descuento'] / 100)

print('Modo activo:', MODO)
print('Shape:', df.shape)

## Paso 2: Exploracion inicial (Big Five)

In [ ]:
display(df.head())
display(df.info())
display(df.describe(include='all').T.head(15))
print('Nulos por columna:')
display(df.isna().sum().sort_values(ascending=False).head(10))

## Paso 3: Limpieza minima consistente

In [ ]:
antes = len(df)
df = df.drop_duplicates().copy()
despues = len(df)
print('Duplicados eliminados:', antes - despues)

# Imputacion simple para numericas
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

print('Nulos restantes:', int(df.isna().sum().sum()))

## Paso 4: Visualización exploratoria clave (univariado + bivariado)

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if num_cols:
    sns.histplot(df[num_cols[0]], kde=True, ax=axes[0], color='#457b9d')
    axes[0].set_title(f'Distribucion de {num_cols[0]}')

if cat_cols:
    top = df[cat_cols[0]].astype(str).value_counts().head(8)
    sns.barplot(x=top.values, y=top.index, ax=axes[1], color='#2a9d8f')
    axes[1].set_title(f'Top categorias: {cat_cols[0]}')

plt.tight_layout()
plt.show()

In [ ]:
if len(num_cols) >= 2:
    corr = df[num_cols].corr()
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, cmap='coolwarm', center=0)
    plt.title('Matriz de correlacion (numericas)')
    plt.show()

## Paso 5: Insights guiados

Completa estos 3 insights con evidencia del notebook:
1. Insight de perfil
2. Insight de relacion entre variables
3. Insight accionable de negocio

In [ ]:
resumen = {
    'filas': len(df),
    'columnas': len(df.columns),
    'nulos_totales': int(df.isna().sum().sum())
}

for c in num_cols[:3]:
    resumen[f'media_{c}'] = round(float(df[c].mean()), 2)

print('Resumen ejecutivo:')
for k, v in resumen.items():
    print(f'- {k}: {v}')

In [ ]:
df.to_csv('curie_eda_resultado.csv', index=False, encoding='utf-8')
print('Exportado: curie_eda_resultado.csv')